# 2.5-2.6 Questions Followup Exercise
Scott Person

## Load and Cleanup

In [0]:
data_path = './data/'

In [0]:
%pip install -q openpyxl
%pip install -q seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

### Load Data
Load both Excel sheets into pandas dataframes

In [0]:
orders_df = pd.read_excel(data_path + 'Global_Superstore_Orders_2012_2015.xlsx', sheet_name='Orders')
people_df = pd.read_excel(data_path + 'Global_Superstore_Orders_2012_2015.xlsx', sheet_name='People')

### Step 1. Setup

#### Print Before Default Setting
Note use of scientific notation

In [0]:
# Commented out to avoid displaying numbers with more than two decimal places. Content is not germane to this assignment.
# orders_df.groupby('Sub-Category', observed=True)['Sales'].sum().sort_values(ascending=False)

#### Notebook Defaults

In [0]:
# Set Pandas to display numbers without scientific notation and with 2 decimal places
pd.options.display.float_format = '{:,.2f}'.format

#### Print After Default Setting
Note no longer using scientific notation

In [0]:
orders_df.groupby('Sub-Category', observed=True)['Sales'].sum().sort_values(ascending=False)

### Print Shape
Print the shape of both dataframes including memory usage for Orders

#### Orders

In [0]:
print(f'Shape: {orders_df.shape}')

print(f'Size: {orders_df.memory_usage(deep=True).sum():,} bytes')

#### People


In [0]:
print(f'Shape: {people_df.shape}')

### Apply Snake

In [0]:
def to_snake(columns):
    'Lowercase, turn anything that is not a letter or digit into _, trim stray underscores.'
    return (pd.Index(columns)
            .str.strip()
            .str.lower()
            .str.replace(r"[^0-9a-z]+", "_", regex=True)
            .str.strip("_"))

In [0]:
people_df.columns = to_snake(people_df.columns)

orders_columns_orig = orders_df.columns
orders_df.columns = to_snake(orders_columns_orig)

pd.DataFrame({"before": orders_columns_orig, "after": orders_df.columns})

### Type every column

This will output a shell of markdown that I can use to start.

In [0]:
def dataframe_schema_markdown(df):
    """Return a markdown table listing each column's name, current dtype,
    and blank columns for a proposed new datatype, and the reason."""
    rows = [
        f"| {i} | {col} | {str(dtype)} |  |  |"
        for i, (col, dtype) in enumerate(df.dtypes.items(), start=1)
    ]
    header = "| # | column_name | datatype | new_datatype | reason |\n|---|---|---|---|---|\n"
    return header + "\n".join(rows)

print(dataframe_schema_markdown(orders_df))

Here's my initial rename recommendations. As with all code/requirements, it's subject to change upon testing.

| # | column_name | datatype | new_datatype | reason |
|---|---|---|---|---|
| 1 | row_id | int64 |int64 |this is a counter; do not use for analysis  |
| 2 | order_id | object |object  |natural key of the table  |
| 3 | order_date | datetime64[ns] |datetime64  |this is a date  |
| 4 | ship_date | datetime64[ns] |datetime64  |this is a date  |
| 5 | ship_mode | object |category  |small, fixed set of values  |
| 6 | customer_id | object |object  |this is an identifier - likely a foreign key  |
| 7 | customer_name | object |object  |high cardinality text; high cardinality data is not conducive to categories; not a common analytical subject  |
| 8 | segment | object |category  |small fixed set of values  |
| 9 | postal_code | float64 |object  |postal codes often have leading zeros which a numerical type will remove; we don't do math on postal codes  |
| 10 | city | object |object  |high cardinality text  |
| 11 | state | object |category |(relatively) fixed set of values with low cardinality  |
| 12 | country | object |category  |as with state, relatively fixed with low cardinality  |
| 13 | region | object |category |low cardinality text; we will likely sort and group on this; this holds for the other category columns as well  |
| 14 | market | object |category  |low cardinality text  |
| 15 | product_id | object |object  |this is an identifier - likely a foreign key  |
| 16 | category | object |category  |low cardinality text  |
| 17 | sub_category | object |category  |low cardinality text  |
| 18 | product_name | object |object  |relatively low cardinality but subject to change  |
| 19 | sales | float64 |int64  |going old school and storing all of my dollar values as int but converting at output to two decimals; this will eliminate floating point math issues; a scale value will be stored in a constant to allow conversion from int back to dollars and cents  |
| 20 | quantity | int64 |int64  |it's a number of things so int makes sense  |
| 21 | discount | float64 |float64  |this is a decimal representation of a percentage  |
| 22 | profit | float64 |int64  |sticking with int for my amounts  |
| 23 | shipping_cost | float64 |int64  |sticking with int for my amounts  |
| 24 | order_priority | object |category - ordered  |small fixed set of values; has a natrual order  |


Apply the datatype assignments in code

In [0]:
MONEY_SCALE = 10000

# --- Convert columns to the datatypes specified in the schema table ---

print(f"Memory usage before: {orders_df.memory_usage(deep=True).sum():,} bytes")

# Category columns (unordered)
for col in ['ship_mode', 'segment', 'state', 'country', 'region',
           'market', 'category', 'sub_category']:
    orders_df[col] = orders_df[col].astype('category')

# Ordered category for order_priority (Low < Medium < High < Critical)
orders_df['order_priority'] = pd.Categorical(
    orders_df['order_priority'],
    categories=['Low', 'Medium', 'High', 'Critical'],
    ordered=True
)

# Postal code: float64 -> object (string), stripping the trailing '.0'
# Padding to 5 digits with leading zeros to preserve values like 02101
orders_df['postal_code'] = (
    orders_df['postal_code']
        .dropna()
        .astype('Int64')      # nullable int so we can go to string cleanly
        .astype('string')
        .str.zfill(5)          # pad with leading zeros to 5 digits
        .reindex(orders_df.index)
)

# Money columns: float64 -> int64 (scaled to eliminate floating-point issues)
for col in ['sales', 'profit', 'shipping_cost']:
    orders_df[col] = (orders_df[col] * MONEY_SCALE).round().astype('int64')

print(f"Memory usage after:  {orders_df.memory_usage(deep=True).sum():,} bytes")
orders_df.dtypes



### Merge and check

In [0]:
print(f"Before merge — orders: {len(orders_df):,}  people: {len(people_df):,}")

orders_merged_df = orders_df.merge(people_df, on='region', how='left', indicator=True, validate='many_to_one')

print(f"After merge  — merged:  {len(orders_merged_df):,}")

print(orders_merged_df.groupby(['region', '_merge'], observed=True).size())

left_only = orders_merged_df.loc[orders_merged_df['_merge'] == 'left_only']

print(f"\nTotal sales (left only): {left_only['sales'].sum() / MONEY_SCALE:,.2f}")

The left_only items are caused by Canada being the region for the orders whereas the respective people have regions of Eastern and Western Canada. This issue is likely a reflection of the source systems and that's where I would address it. I would consult with the business and either assign a single person to Canada or assign the orders to the Canadian regions based on their province. The region assignment is buried somewhere in their business system and depending on complexity that's where I'd look.

If this data is all that I have, I'd simply fix the orders data based on the province. The issue with doing it here is this fix is ephemeral. The next pull will need the fix again.

## Section 2.5-2.6 Questions Work Starts Here

### 1. Three numbers, plainly

In [0]:
# Sales and Profit summary statistics (convert scaled ints back to dollars)

sales_mean = orders_merged_df['sales'].mean() / MONEY_SCALE
sales_median = orders_merged_df['sales'].median() / MONEY_SCALE
sales_std = orders_merged_df['sales'].std() / MONEY_SCALE

profit_mean = orders_merged_df['profit'].mean() / MONEY_SCALE
profit_median = orders_merged_df['profit'].median() / MONEY_SCALE
profit_std = orders_merged_df['profit'].std() / MONEY_SCALE

print(f"Sales have a mean of ${sales_mean:,.2f}, a median of ${sales_median:,.2f}, and a standard deviation of ${sales_std:,.2f}.")
print(f"Profit has a mean of ${profit_mean:,.2f}, a median of ${profit_median:,.2f}, and a standard deviation of ${profit_std:,.2f}.")

In [0]:
# Coefficient of variation (std / mean) for sales and profit

sales_cv = orders_merged_df['sales'].std() / orders_merged_df['sales'].mean()
profit_cv = orders_merged_df['profit'].std() / orders_merged_df['profit'].mean()

print(f"Sales coefficient of variation:  {sales_cv:,.2f}")
print(f"Profit coefficient of variation: {profit_cv:,.2f}")

For both columns, the standard deviation is quite high. Sales with a coefficient of varaiation of 1.98 is high enough that we can assume significant skew given the mean and median relationship this would be right skew. 

Profit can be negative and therefore the coefficient of variation would potentially be misleading.

### 2. Test the rule

#### Check if the sales data has a normal distribution

In [0]:
def share_within(series, k):
    'Share of the values that are within k standard deviations of the mean.'
    mean, std = series.mean(), series.std()
    return series.between(mean - k * std, mean + k * std).mean()


def rule_check(series):
    'Compare a column against the 68-95-99.7 rule.'
    return pd.DataFrame({
        "within":          ["1 std", "2 std", "3 std"],
        "bell-curve rule": [0.68, 0.95, 0.997],
        "this column":     [round(share_within(series, k), 3) for k in (1, 2, 3)],
    })


rule_check(orders_merged_df['sales'])

#### Show a histogram of sales and interpret

In [0]:
# Convert scaled int back to dollars for the plot
sales_dollars = orders_merged_df['sales'] / MONEY_SCALE

plt.figure(figsize=(10, 6))
sns.histplot(sales_dollars, bins=50)
plt.xlabel('Sales ($)')
plt.ylabel('Count')
plt.title('Distribution of Sales')
plt.show()

In [0]:
# Median and IQR for Sales and Profit (convert scaled ints back to dollars)

sales_q1, sales_q3 = orders_merged_df['sales'].quantile([0.25, 0.75]) / MONEY_SCALE
sales_iqr = sales_q3 - sales_q1

print(f"Sales have a median of ${sales_median:,.2f} and an IQR of ${sales_iqr:,.2f} (Q1=${sales_q1:,.2f}, Q3=${sales_q3:,.2f}).")

The histogram agrees with my interpretation of the mean and median relationship above where we have a very strong right skew to the sales data. This means that the data is not normally distributed and a 68-95-99.7 distribution for 1, 2, and 3 standard deviations will not hold. For a strongly skewed dataset we often use the median and the IQR (calculated immediately above).

### 3. Flag, then decide

In [0]:
# Find and count records whose sales z-score exceeds 3 (scale cancels out)

sales_mean = orders_merged_df['sales'].mean()
sales_std = orders_merged_df['sales'].std()

z_scores = (orders_merged_df['sales'] - sales_mean) / sales_std

z_outliers = orders_merged_df.loc[z_scores.abs() > 3]

print(f"Records with |z| > 3: {len(z_outliers)}")



In [0]:
def iqr_fences(series):
    'The box plot fences: 1.5 IQR below the first quartile and above the third.'
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


low, high = iqr_fences(sales_dollars)

flagged = sales_dollars[~sales_dollars.between(low, high)]

print(f"fences: {low:.2f} to {high:.2f}")
print(f"total flagged: {len(flagged)}")

#### Ten largest orders (by sales)
It is not entirely clear if the question is asking for order lines or aggregated orders. Our questions have dealt with individual lines but the aggregation appears to be the strict interpretation of the question. I'll show both. 

In [0]:
# Ten largest records by sales (convert scaled ints back to dollars for display)

top10 = orders_merged_df.nlargest(10, 'sales').copy()

for col in ['sales', 'profit', 'shipping_cost']:
    top10[col] = top10[col] / MONEY_SCALE

top10[['order_id', 'order_date', 'customer_name', 'region', 'product_name', 'sales', 'quantity', 'discount', 'profit']]

In [0]:
# Ten largest orders by total sales (aggregate lines within each order_id, then convert scaled ints back to dollars)

agg_map = {
    'sales': 'sum',
    'profit': 'sum',
    'shipping_cost': 'sum',
    'quantity': 'sum',
    'order_date': 'first',
    'customer_name': 'first',
    'region': 'first',
}

orders_agg = orders_merged_df.groupby('order_id', as_index=False).agg(agg_map)

top10 = orders_agg.nlargest(10, 'sales').copy()

for col in ['sales', 'profit']:
    top10[col] = top10[col] / MONEY_SCALE

top10[['order_id', 'order_date', 'customer_name', 'region', 'sales', 'quantity', 'profit']]

These appear to be real orders. The quantities are plausible as are the total dollar amounts. The largest amount by sales represents a loss but this can be explained by the discount given. Keeping legitimate extremes of data comes with some cost. While large order activity is retained, the mean and standard deviation are particularly sensitive to these values. The cost of removing them would mean the loss of valuable analytical data.

### 4. Compare groups


In [0]:
# Mean, median, and standard deviation of profit by category
# (convert scaled ints back to dollars, round to 2 decimals, sort by std descending)

profit_by_cat = (
    orders_merged_df
        .groupby('category', observed=True)['profit']
        .agg(['mean', 'median', 'std'])
        / MONEY_SCALE
)

profit_by_cat = profit_by_cat.round(2).sort_values('std', ascending=False, inplace=False)

profit_by_cat

Based on standard deviation, technology is the least predictable in absolute dollar terms. Someone running that part of the business should expect less consistency in profitability. Spending some time investigating this unpredictability would be prudent.

I used the sort_values method of a dataframe to sort by standard deviation (descending in this case). The method has the option to change the original table by using inplace=True. I generally find the use of this to somewhat less clear than assigning back to the same name so I elected to explicitly use inplace=False (also the default behavior). Because inplace=False leaves the original dataframe object unchanged, I assigned the sorted result back to the same variable name. The variable then refers to the sorted table. This can all be found in the documentation for sort_values.